# Step 1: Raw data loading and parsing

We extract HUST bearing metadata from raw dataset folder structure, then loads and parses a bearing Excel data file (.xls) into pd.DataFrame, and saves as csv file.

The resulting data is called the 'parsed' data.

In [120]:
# Load the "autoreload" extension so that code can change
%load_ext autoreload
# Always reload modules so that as you change code in src, it gets loaded
%autoreload 2

import os
from io import StringIO
from copy import deepcopy
import json
import yaml
import re
import gc

import numpy as np
import pandas as pd
from scipy.io import loadmat

import matplotlib.pyplot as plt

from data_parse import get_bearing_metadatas, parse_dataset, RAW_DATA_DIR
# from blob_utils import get_container_client, download_blob, download_container, save_file_to_blob, save_folder_to_container

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Extract metadata from raw data folder structure

In [67]:
metadatas = get_bearing_metadatas(output_dir='./')

## Data loading and parsing

Extract bearing metadata, parse matlab files to csv files

In [122]:
parsed_metadatas = parse_dataset()

In [126]:
parsed_metadatas[:3]

[{'raw_filename': '0.5X_B_20Hz.xls',
  'raw_data_path': './data/0.5X_B_20Hz.xls',
  'fault_label': 'M_B',
  'fault_severity': 'medium',
  'fault_location': 'B',
  'bearing_speed': '20Hz',
  'sampling_freq_khz': 25.641,
  'df_length': 262144,
  'df_columns': ['timestamp', 'speed', 'x', 'y', 'z'],
  'parsed_data_path': './data_parsed/csv/0.5X_B_20Hz.csv'},
 {'raw_filename': '0.5X_B_25Hz.xls',
  'raw_data_path': './data/0.5X_B_25Hz.xls',
  'fault_label': 'M_B',
  'fault_severity': 'medium',
  'fault_location': 'B',
  'bearing_speed': '25Hz',
  'sampling_freq_khz': 25.641,
  'df_length': 262144,
  'df_columns': ['timestamp', 'speed', 'x', 'y', 'z'],
  'parsed_data_path': './data_parsed/csv/0.5X_B_25Hz.csv'},
 {'raw_filename': '0.5X_B_30Hz.xls',
  'raw_data_path': './data/0.5X_B_30Hz.xls',
  'fault_label': 'M_B',
  'fault_severity': 'medium',
  'fault_location': 'B',
  'bearing_speed': '30Hz',
  'sampling_freq_khz': 25.641,
  'df_length': 262144,
  'df_columns': ['timestamp', 'speed', 'x', 

## Quick metadata statistics

In [127]:
df_parsed_metadatas = pd.DataFrame(parsed_metadatas).sort_values('fault_location')

In [128]:
df_parsed_metadatas

,raw_filename,raw_data_path,fault_label,fault_severity,fault_location,bearing_speed,sampling_freq_khz,df_length,df_columns,parsed_data_path
0,0.5X_B_20Hz.xls,./data/0.5X_B_20Hz.xls,M_B,medium,B,20Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/0.5X_B_20Hz.csv
54,B_VS_0_40_0Hz.xls,./data/B_VS_0_40_0Hz.xls,S_B,severe,B,VS_0_40_0Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/B_VS_0_40_0Hz.csv
53,B_80Hz.xls,./data/B_80Hz.xls,S_B,severe,B,80Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/B_80Hz.csv
52,B_75Hz.xls,./data/B_75Hz.xls,S_B,severe,B,75Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/B_75Hz.csv
51,B_70Hz.xls,./data/B_70Hz.xls,S_B,severe,B,70Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/B_70Hz.csv
...,...,...,...,...,...,...,...,...,...,...
38,0.5X_O_60Hz.xls,./data/0.5X_O_60Hz.xls,M_O,medium,O,60Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/0.5X_O_60Hz.csv
37,0.5X_O_40Hz.xls,./data/0.5X_O_40Hz.xls,M_O,medium,O,40Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/0.5X_O_40Hz.csv
36,0.5X_O_35Hz.xls,./data/0.5X_O_35Hz.xls,M_O,medium,O,35Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/0.5X_O_35Hz.csv
34,0.5X_O_25Hz.xls,./data/0.5X_O_25Hz.xls,M_O,medium,O,25Hz,25.641,262144,"[timestamp, speed, x, y, z]",./data_parsed/csv/0.5X_O_25Hz.csv


In [129]:
df_parsed_metadatas['fault_label'].value_counts()

M_B    11
S_B    11
M_C    11
S_C    11
H      11
M_I    11
S_I    11
S_O    11
M_O    11
Name: fault_label, dtype: int64

In [131]:
df_parsed_metadatas.groupby(['fault_label'])['bearing_speed'].value_counts()

fault_label  bearing_speed
H            20Hz             1
             25hz             1
             30Hz             1
             35Hz             1
             40Hz             1
                             ..
S_O          65Hz             1
             70Hz             1
             75Hz             1
             80Hz             1
             VS_0_40_0Hz      1
Name: bearing_speed, Length: 99, dtype: int64